**Navigation** : [carnet source Python](Coalitions-04-Munkres-Affectation-Python.ipynb) | [<< Coalitions-03 Echange-de-Reins](Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) | [Index](../README.md) | [Coalitions-05 Lean-Affectation >>](Coalitions-05-Lean-Affectation-Lean.ipynb)

# Coalitions-04 (C#) — L'algorithme de Kuhn-Munkres : affectation optimale et dualité

Jumeau C# de [Coalitions-04 — l'algorithme de Kuhn-Munkres (Python)](Coalitions-04-Munkres-Affectation-Python.ipynb) : même fil pédagogique, mêmes entrées, même arithmétique exacte — rejouée ici en `BigInteger` (BCL seule). La confrontation à la référence SOTA `scipy.optimize.linear_sum_assignment` du carnet Python devient ici une confrontation à la **force brute interne** : l'énumération exhaustive des permutations, référence exacte par construction.

> **Hommage.** James R. Munkres, professeur émérite au MIT, s'est éteint le 30 juillet 2026 à 95 ans.
> Si son nom est familier des générations d'étudiants par son manuel de topologie (*Topology*,
> 1975 — le « Munkres » des cours 18.901), il est pour nous le co-éponyme de l'**algorithme de
> Kuhn-Munkres** — la méthode hongroise pour le problème d'affectation, dont il a établi en 1957
> la première analyse de complexité *fortement polynomiale*. Ce notebook travaille son théorème.

> **Écho dans le dépôt.** La méthode tourne en production pédagogique dans
> [Sudoku-15 — Solveurs probabilistes (C#)](../../Sudoku/Sudoku-15-Infer-CSharp.ipynb) :
> le solveur v4 (max-product sur facteurs AllDiff d'arité 9) appelle Kuhn-Munkres (`Hungarian`)
> pour extraire son affectation à chaque message.

## 1. Le problème d'affectation

$n$ agents, $n$ tâches, une matrice de coûts $c_{ij}$ (le coût pour que l'agent $i$ fasse la tâche $j$).
Une **affectation** (matching parfait) $\sigma$ est une permutation : chaque agent reçoit exactement
une tâche, chaque tâche exactement un agent. Le problème :

$$\min_{\sigma \in S_n} \sum_{i=1}^{n} c_{i,\sigma(i)}$$

L'énumération est hors de question : $|S_n| = n!$ — à $n = 12$ cela dépasse déjà 479 millions de
permutations. L'algorithme de Kuhn-Munkres résout le problème en $O(n^3)$.

**Fil conducteur du notebook** : nous implémentons l'algorithme *from scratch* (labels, graphe
d'égalité, arbres hongrois), le confrontons à la **force brute exhaustive** — la référence exacte
de l'énumération, qui tient ici parce que $n$ reste petit — puis nous lisons le résultat par la
**dualité linéaire** — et cette lecture ouvre la porte à la
théorie des jeux : le **jeu d'affectation de Shapley-Shubik**, dont le cœur coïncide exactement
avec les solutions duales optimales.

In [1]:
// Arithmetique exacte du carnet : BigInteger partout (System.Numerics), BCL seule.
// Aucun flottant, aucune egalite flottante : chaque valeur ancree leve une assertion si elle differe.
using System;
using System.Collections.Generic;
using System.Linq;
using System.Numerics;

// Verifier : assertion EXECUTABLE -- leve si la condition est fausse.
static void Verifier(bool cond, string message)
{
    if (!cond) throw new Exception($"ECHEC D'ASSERTION : {message}");
    Console.WriteLine($"Assertion OK : {message}");
}

// L'exemple fil rouge : 4 agents, 4 taches (couts positifs, exemple deterministe).
// CITE : identique a la matrice du carnet source Python (cellule 2).
BigInteger[,] C = {
    { 9, 2, 7, 8 },
    { 6, 4, 3, 7 },
    { 5, 8, 1, 8 },
    { 7, 6, 9, 4 },
};
int n = C.GetLength(0);

static BigInteger Factorial(int k)
{
    var r = BigInteger.One;
    for (int i = 2; i <= k; i++) r *= i;
    return r;
}

Console.WriteLine("Matrice de couts C :");
for (int i = 0; i < n; i++)
    Console.WriteLine("  [" + string.Join(", ", Enumerable.Range(0, n).Select(j => C[i, j].ToString())) + "]");
Console.WriteLine();
Console.WriteLine($"Taille de l'espace de recherche : {n}! = {Factorial(n)} permutations");

Verifier(n == 4, "la matrice fil rouge est 4x4");
Verifier(Factorial(n) == 24, "4! = 24 permutations");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Matrice de couts C :


  [9, 2, 7, 8]


  [6, 4, 3, 7]


  [5, 8, 1, 8]


  [7, 6, 9, 4]


Taille de l'espace de recherche : 4! = 24 permutations


Assertion OK : la matrice fil rouge est 4x4


Assertion OK : 4! = 24 permutations


### Lecture de l'exemple

La matrice $C$ dit par exemple que l'agent 0 fait la tâche 1 pour 2 (bon marché) mais la tâche 0
pour 9 (cher). Une heuristique gloutonne — prendre à chaque ligne son minimum — donnerait
2 + 3 + 1 + 4 = 10, mais les colonnes 1 et 2 seraient disputées : le glouton ne produit pas une
permutation. Le vrai problème est de trouver l'équilibre entre les lignes. Vérifions d'abord par
force brute (possible à $n=4$), pour avoir une vérité de référence.

In [2]:
// Force brute : toutes les permutations (n=4 seulement !)
// Valeur d'une affectation sigma : somme des couts c[i, sigma[i]].
static BigInteger ValeurBrute(BigInteger[,] M, int[] sigma)
{
    BigInteger s = 0;
    for (int i = 0; i < M.GetLength(0); i++) s += M[i, sigma[i]];
    return s;
}

// Enumeration lexicographique des permutations de 0..k-1 (iteratif, BCL seule).
// L'ordre lexicographique reproduit le min(...) du carnet Python sur les couples (valeur, liste).
static IEnumerable<int[]> Permutations(int k)
{
    var p = Enumerable.Range(0, k).ToArray();
    while (true)
    {
        yield return (int[])p.Clone();
        int i = k - 2;                              // plus grand i avec p[i] < p[i+1]
        while (i >= 0 && p[i] >= p[i + 1]) i--;
        if (i < 0) yield break;                     // derniere permutation
        int j = k - 1;                              // plus grand j > i avec p[j] > p[i]
        while (p[j] <= p[i]) j--;
        (p[i], p[j]) = (p[j], p[i]);
        Array.Reverse(p, i + 1, k - i - 1);
    }
}

BigInteger bruteVal = BigInteger.Zero;
int[] bruteArg = null;
int nbPermTestees = 0;
foreach (var p in Permutations(n))
{
    nbPermTestees++;
    var v = ValeurBrute(C, p);
    if (bruteArg == null || v < bruteVal)           // strict : la premiere minimale en ordre lexicographique
    {
        bruteVal = v;
        bruteArg = (int[])p.Clone();
    }
}
Console.WriteLine($"Optimum par force brute : {bruteVal}  (permutation [{string.Join(", ", bruteArg)}])");
Console.WriteLine($"Permutations testees : {nbPermTestees}");

Verifier(bruteVal == 13, "optimum force brute == 13");
Verifier(bruteArg.SequenceEqual(new[] { 1, 0, 2, 3 }), "permutation optimale == [1, 0, 2, 3]");
Verifier(nbPermTestees == 24, "24 permutations testees");

Optimum par force brute : 13  (permutation [1, 0, 2, 3])


Permutations testees : 24


Assertion OK : optimum force brute == 13


Assertion OK : permutation optimale == [1, 0, 2, 3]


Assertion OK : 24 permutations testees


## 2. L'implémentation pédagogique de Kuhn-Munkres

L'algorithme maintient des **potentiels duaux** (les « labels ») $u_i$ pour les lignes et $v_j$
pour les colonnes, avec la contrainte de **réalisabilité duale** :

$$u_i + v_j \le c_{ij} \quad \text{pour tout } (i, j)$$

Le **graphe d'égalité** est le sous-graphe des arêtes *saturées* : $u_i + v_j = c_{ij}$.
Un matching n'utilisant que des arêtes d'égalité et couvrant tout le monde est **optimal** —
c'est le théorème de dualité (le matching est une solution primale réalisable, les potentiels
une solution duale réalisable, et ils ont la même valeur : aucun gap).

Tant qu'un matching parfait n'existe pas dans le graphe d'égalité, l'algorithme construit un
**arbre hongrois** (alternant depuis une ligne non appariée) et **resserre** les labels du
montant minimal $\delta$ qui fait apparaître une nouvelle arête d'égalité. C'est ce $\delta$
qui garantit la terminaison et le temps polynomial.

In [3]:
// Algorithme de Kuhn-Munkres (methode hongroise), version pedagogique -- port exact du Python.
// Travaille en ARITHMETIQUE ENTIERS EXACTE (BigInteger) : les resserrements successifs de labels
// accumuleraient en flottants une erreur d'arrondi qui casse le test d'egalite u_i + v_j == c_ij
// -- et avec lui la terminaison. Sur entiers, l'egalite est exacte et l'algorithme termine.
// Chaque iteration construit UNE FOIS l'arbre hongrois (BFS depuis la ligne non appariee i0) :
// decJ[j] memorise la ligne qui a decouvert la colonne j. Si le BFS atteint une colonne libre,
// le chemin augmentant se remonte par decJ et matchRow ; sinon, l'arbre fournit le delta.
// Retourne (valeur, affectation, u, v) avec u_i + v_j <= c_ij pour tout (i, j).
static (BigInteger valeur, int[] assign, BigInteger[] u, BigInteger[] v) KuhnMunkres(BigInteger[,] C)
{
    int n = C.GetLength(0);
    var u = new BigInteger[n];
    var v = new BigInteger[n];
    for (int i = 0; i < n; i++)
    {
        u[i] = C[i, 0];                             // dual realisable : u_i + 0 <= c_ij
        for (int j = 1; j < n; j++) if (C[i, j] < u[i]) u[i] = C[i, j];
    }

    var matchCol = new int[n]; Array.Fill(matchCol, -1);   // colonne j -> ligne appariee
    var matchRow = new int[n]; Array.Fill(matchRow, -1);   // ligne i -> colonne appariee

    for (int i0 = 0; i0 < n; i0++)
    {
        while (matchRow[i0] == -1)
        {
            // BFS alternant depuis i0 dans le graphe d'egalite.
            var decJ = new Dictionary<int, int>();        // colonne -> ligne qui l'a decouverte
            var lignes = new HashSet<int> { i0 };
            var colonnes = new HashSet<int>();
            var file = new Queue<int>(); file.Enqueue(i0);
            int jLibre = -1;
            while (file.Count > 0 && jLibre == -1)
            {
                int i = file.Dequeue();
                for (int j = 0; j < n; j++)
                {
                    if (colonnes.Contains(j) || u[i] + v[j] != C[i, j]) continue;
                    decJ[j] = i;
                    colonnes.Add(j);
                    if (matchCol[j] == -1) { jLibre = j; break; }
                    lignes.Add(matchCol[j]);
                    file.Enqueue(matchCol[j]);
                }
            }
            if (jLibre >= 0)
            {
                // Remonter le chemin augmentant : colonne libre <- decJ -> ligne appariee
                // <- matchRow -> colonne ... jusqu'a i0.
                int j = jLibre;
                int i = decJ[j];
                while (true)
                {
                    int jSuiv = matchRow[i];              // ancienne colonne de la ligne i
                    matchCol[j] = i;
                    matchRow[i] = j;
                    if (i == i0) break;
                    j = jSuiv;
                    i = decJ[j];
                }
            }
            else
            {
                // Arbre bloque : resserrement par delta, la plus petite marge sur les aretes
                // sortantes (ligne dans l'arbre, colonne hors arbre) -- strictement positive
                // par construction de l'arbre.
                BigInteger delta = BigInteger.Zero;
                bool premiere = true;
                foreach (var i in lignes)
                    for (int j = 0; j < n; j++)
                    {
                        if (colonnes.Contains(j)) continue;
                        var marge = C[i, j] - u[i] - v[j];
                        if (premiere || marge < delta) { delta = marge; premiere = false; }
                    }
                foreach (var i in lignes) u[i] += delta;
                foreach (var j in colonnes) v[j] -= delta;
            }
        }
    }
    var trie = (int[])matchRow.Clone(); Array.Sort(trie);
    for (int t = 0; t < n; t++)
        if (trie[t] != t) throw new Exception("pas un matching parfait");
    BigInteger valeur = 0;
    for (int i = 0; i < n; i++) valeur += C[i, matchRow[i]];
    return (valeur, (int[])matchRow.Clone(), u, v);
}

var (valKm, assignKm, u, v) = KuhnMunkres(C);
Console.WriteLine($"Kuhn-Munkres from scratch : valeur = {valKm}, affectation = [{string.Join(", ", assignKm)}]");

Verifier(valKm == 13, "valeur Kuhn-Munkres == 13");
Verifier(assignKm.SequenceEqual(new[] { 1, 0, 2, 3 }), "affectation Kuhn-Munkres == [1, 0, 2, 3]");

Kuhn-Munkres from scratch : valeur = 13, affectation = [1, 0, 2, 3]


Assertion OK : valeur Kuhn-Munkres == 13


Assertion OK : affectation Kuhn-Munkres == [1, 0, 2, 3]


In [4]:
// Confrontation a la reference : la FORCE BRUTE INTERNE (enumeration exacte).
// Le carnet Python confronte scipy.optimize.linear_sum_assignment (Jonker-Volgenant) ;
// le jumeau C#, sans appel externe, confronte l'enumeration des permutations -- une
// reference exacte par construction sur ces tailles.
Console.WriteLine($"Force brute (reference)    : valeur = {bruteVal}, affectation = [{string.Join(", ", bruteArg)}]");
Console.WriteLine($"Coincidence from scratch == reference : {valKm == bruteVal && assignKm.SequenceEqual(bruteArg)}");

// Optimum par force brute pour une matrice quelconque (utilise aussi plus bas).
static BigInteger OptimumBrute(BigInteger[,] M)
{
    int k = M.GetLength(0);
    BigInteger best = BigInteger.Zero;
    bool premiere = true;
    foreach (var p in Permutations(k))
    {
        var val = ValeurBrute(M, p);
        if (premiere || val < best) { best = val; premiere = false; }
    }
    return best;
}

// Verification multi-instances (deterministe, graine fixee) : 50 matrices 6x6
var rng = new Random(42);
BigInteger maxEcart = BigInteger.Zero;
for (int k = 0; k < 50; k++)
{
    var M = new BigInteger[6, 6];
    for (int i = 0; i < 6; i++)
        for (int j = 0; j < 6; j++) M[i, j] = rng.Next(0, 50);
    var (vk, ak, _, _) = KuhnMunkres(M);
    var vBrute = OptimumBrute(M);
    maxEcart = BigInteger.Max(maxEcart, BigInteger.Abs(vk - vBrute));
}
Console.WriteLine($"50 instances aleatoires 6x6 : max |KM - brute| = {maxEcart}");

Verifier(valKm == bruteVal && assignKm.SequenceEqual(bruteArg), "KM == force brute (valeur et affectation) sur le fil rouge");
Verifier(maxEcart == 0, "max |KM - brute| == 0 sur 50 instances 6x6");

Force brute (reference)    : valeur = 13, affectation = [1, 0, 2, 3]


Coincidence from scratch == reference : True


50 instances aleatoires 6x6 : max |KM - brute| = 0


Assertion OK : KM == force brute (valeur et affectation) sur le fil rouge


Assertion OK : max |KM - brute| == 0 sur 50 instances 6x6


### Lecture du résultat

Sur le fil rouge, KM from scratch et la force brute retournent **la même valeur (13) et la même
affectation [1, 0, 2, 3]** — coïncidence exacte. Sur 50 instances aléatoires 6×6 (graine 42,
coûts entiers) : **max |KM − brute| = 0** — cinquante accords parfaits. L'accord porte sur
la valeur ; plusieurs affectations peuvent atteindre 13 sur d'autres instances, ici même la
permutation coïncide.

### Ce que la confrontation établit

Sur l'exemple fil rouge comme sur 50 instances aléatoires, l'implémentation pédagogique et la
référence coïncident exactement — même valeur, même affectation. La référence du jumeau Python
est `scipy.optimize.linear_sum_assignment` (une variante jonction-flots de la même idée
algorithmique — Jonker-Volgenant) ; la référence du jumeau C# est **l'énumération exhaustive**,
exacte par construction tant que $n!$ reste énumérable. Le point pédagogique est le même des
deux côtés : **la force brute et la bibliothèque donnent la valeur, mais pas les potentiels
duaux $(u, v)$** — notre version les expose, et ce sont eux qui portent toute la suite du
notebook.

## 3. La lecture par dualité linéaire

Le problème d'affectation est un programme linéaire (en variables $x_{ij} \in \{0,1\}$,
relaxé en $[0,1]$ — l'intégralité est automatique car la matrice est unimodulaire) :

$$\min \sum_{ij} c_{ij} x_{ij} \quad \text{s.c.} \quad \sum_j x_{ij} = 1 \ \forall i, \quad \sum_i x_{ij} = 1 \ \forall j$$

Son **dual** a exactement la forme de nos potentiels :

$$\max \sum_i u_i + \sum_j v_j \quad \text{s.c.} \quad u_i + v_j \le c_{ij} \ \forall (i,j)$$

La théorie LP promet : toute solution primale réalisable et toute solution duale réalisable
avec **valeurs égales** sont toutes deux **optimales** (dualité faible + gap nul). Vérifions
numériquement que notre run le réalise.

In [5]:
// Verifications de dualite sur le run precedent
// (1) realisabilite duale : u_i + v_j <= c_ij partout
BigInteger maxViol = BigInteger.Zero;
for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++)
    {
        var viol = u[i] + v[j] - C[i, j];
        if (viol > maxViol) maxViol = viol;
    }
Console.WriteLine($"(1) Realisabilite duale  : max(u_i + v_j - c_ij) = {maxViol} (<= 0 attendu)");

// (2) egalite primale-duale : somme des potentiels == valeur du matching
// (pas de Sum() sur BigInteger : Aggregate explicite)
BigInteger valDual = u.Aggregate(BigInteger.Zero, (a, b) => a + b)
                 + v.Aggregate(BigInteger.Zero, (a, b) => a + b);
Console.WriteLine($"(2) Valeur primale (matching) = {valKm}");
Console.WriteLine($"    Valeur duale  (u.sum + v.sum) = {valDual}");
Console.WriteLine($"    Gap de dualite = {BigInteger.Abs(valKm - valDual)} (0 attendu)");

// (3) les aretes du matching sont toutes des aretes d'egalite
bool surEg = Enumerable.Range(0, n).All(i => u[i] + v[assignKm[i]] == C[i, assignKm[i]]);
Console.WriteLine($"(3) Toutes les aretes du matching sont d'egalite : {surEg}");

Verifier(maxViol <= 0, "realisabilite duale : max(u_i + v_j - c_ij) <= 0");
Verifier(valKm == 13 && valDual == 13, "valeurs primale et duale == 13");
Verifier(valKm == valDual, "gap de dualite nul");
Verifier(surEg, "les 4 aretes du matching sont des aretes d'egalite");

(1) Realisabilite duale  : max(u_i + v_j - c_ij) = 0 (<= 0 attendu)


(2) Valeur primale (matching) = 13


    Valeur duale  (u.sum + v.sum) = 13


    Gap de dualite = 0 (0 attendu)


(3) Toutes les aretes du matching sont d'egalite : True


Assertion OK : realisabilite duale : max(u_i + v_j - c_ij) <= 0


Assertion OK : valeurs primale et duale == 13


Assertion OK : gap de dualite nul


Assertion OK : les 4 aretes du matching sont des aretes d'egalite


### Lecture du résultat

Le triple test LP est **vert** : (1) faisabilité duale exacte — la plus grande violation de
$u_i + v_j \le c_{ij}$ est **0** ; (2) valeur primale = valeur duale = **13**, gap de
dualité **0** ; (3) les 4 arêtes du matching sont toutes des arêtes d'égalité. Par la dualité
forte, ce couple (matching, potentiels) est **certifié optimal** : aucun solveur extérieur ne
peut faire mieux que 13 — et nous le savons sans lui faire confiance, le certificat vit dans
le notebook.

## 4. Le pont théorie des jeux : le jeu d'affectation de Shapley-Shubik

Le théorème de Shapley-Shubik (1971) vit sur un jeu de **surplus**. On convertit les coûts en
valeurs : $\alpha_{ij} = M - c_{ij}$ avec $M = \max C$. Maximiser le surplus total revient à
minimiser le coût total (la même permutation est optimale) — le surplus optimal vaut $nM - 13$.

Le jeu coopératif : chaque agent $i$ et chaque vendeur de tâche $j$ (le joueur $n + j$) sont des
joueurs ; une paire $(i, j)$ laissée seule génère un surplus $\alpha_{ij}$. La **valeur** d'une
coalition $S$ est le meilleur surplus que $S$ réalise en interne : l'affectation optimale dans
le rectangle agents($S$) × vendeurs($S$). Une **imputation** répartit le surplus optimal entre
les $2n$ joueurs ; elle est dans le **cœur** si elle est *efficiente* ($\sum = $ valeur de la
grande coalition) et si **aucune coalition ne peut faire mieux seule** : part($S$) $\ge$
valeur($S$) pour tout $S$.

Le théorème : **le cœur du jeu d'affectation coïncide avec l'ensemble des solutions duales
optimales** du LP de surplus. Concrètement, nos potentiels KM $(u, v)$ — qui satisfont
$u_i + v_j \le c_{ij}$ — donnent une imputation de cœur par translation :
$\mathrm{alloc}_i = U - u_i$ et $\mathrm{alloc}_{n+j} = (M - U) - v_j$, pour n'importe quel
$U \in [\max u,\ M - \max v]$ (l'intervalle garantit des parts non négatives). Chaque
inégalité de paire du cœur se lit alors $\mathrm{alloc}_i + \mathrm{alloc}_{n+j} \ge
\alpha_{ij}$ — la réalisabilité duale, mot pour mot.

C'est l'un des résultats les plus beaux de la théorie coopérative : une brique purement
combinatoire (le cœur, défini par une exponentielle d'inégalités de coalition) se révèle être
un objet LP (le polytope dual, fini et calculable). Le vérifier numériquement.

In [6]:
// Le jeu d'affectation de Shapley-Shubik sur notre exemple, en version surplus.
// alpha_ij = M - c_ij : maximiser le surplus = minimiser le cout (meme optimum).
// Joueurs : n agents + n vendeurs (proprietaires des taches).
// Valeur d'une coalition S = meilleur surplus interne : affectation optimale
// dans le rectangle agents(S) x vendeurs(S).
// Coeur : part(S) >= valeur(S) pour toute coalition S (aucune ne peut bloquer).

BigInteger M = BigInteger.Zero;
for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++)
        if (C[i, j] > M) M = C[i, j];

var alpha = new BigInteger[n, n];
for (int i = 0; i < n; i++)
    for (int j = 0; j < n; j++) alpha[i, j] = M - C[i, j];

// Surplus optimal : max sur les permutations de sum alpha (force brute, independante de KM).
BigInteger vSurplus = BigInteger.Zero;
bool premiereSurplus = true;
foreach (var p in Permutations(n))
{
    BigInteger s = 0;
    for (int i = 0; i < n; i++) s += alpha[i, p[i]];
    if (premiereSurplus || s > vSurplus) { vSurplus = s; premiereSurplus = false; }
}
Console.WriteLine($"Surplus optimal : {vSurplus}  (= n*M - 13 = {n * M - 13})");

// Imputation de coeur derivee des potentiels KM (u, v) : translation par U.
BigInteger uMax = u.Max();
BigInteger vMax = v.Max();
Console.WriteLine($"Parts non negatives pour tout U dans [{uMax}, {M - vMax}] ; on prend U = {uMax}");
BigInteger U = uMax;
var alloc = new Dictionary<int, BigInteger>();
for (int i = 0; i < n; i++) alloc[i] = U - u[i];
for (int j = 0; j < n; j++) alloc[n + j] = (M - U) - v[j];

// Sous-ensembles de taille k d'un ensemble d'indices, en ordre lexicographique (BCL seule).
static IEnumerable<int[]> Combinations(int[] items, int k)
{
    int m = items.Length;
    var idx = Enumerable.Range(0, k).ToArray();
    while (true)
    {
        yield return idx.Select(t => items[t]).ToArray();
        int i = k - 1;
        while (i >= 0 && idx[i] == m - k + i) i--;
        if (i < 0) yield break;
        idx[i]++;
        for (int j = i + 1; j < k; j++) idx[j] = idx[j - 1] + 1;
    }
}

// Meilleur surplus interne d'une coalition S : matching partiel optimal dans le rectangle
// agents(S) x vendeurs(S) (chacun apparie au plus une fois, certains seuls -- rectangles).
static BigInteger ValeurCoalition(int[] S, BigInteger[,] alpha, int n)
{
    var A = S.Where(p => p < n).OrderBy(p => p).ToArray();        // agents de la coalition
    var B = S.Where(p => p >= n).Select(p => p - n).OrderBy(p => p).ToArray();   // vendeurs
    if (A.Length == 0 || B.Length == 0) return 0;
    BigInteger best = 0;
    var utilises = new bool[B.Length];
    void Rec(int idx, BigInteger acc)
    {
        if (idx == A.Length) { if (acc > best) best = acc; return; }
        Rec(idx + 1, acc);                                        // cet agent reste seul
        for (int j = 0; j < B.Length; j++)
        {
            if (utilises[j]) continue;
            utilises[j] = true;
            Rec(idx + 1, acc + alpha[A[idx], B[j]]);
            utilises[j] = false;
        }
    }
    Rec(0, 0);
    return best;
}

int coalitionsTestees = 0;
int violations = 0;
var joueurs = Enumerable.Range(0, 2 * n).ToArray();
for (int taille = 1; taille < 2 * n; taille++)
    foreach (var S in Combinations(joueurs, taille))
    {
        coalitionsTestees++;
        BigInteger part = 0;
        foreach (var p in S) part += alloc[p];
        var vs = ValeurCoalition(S, alpha, n);
        if (part < vs) violations++;
    }
Console.WriteLine($"Coalitions testees : {BigInteger.Pow(2, 2 * n) - 2}");
Console.WriteLine($"Violations du coeur par l'imputation KM : {violations}");
BigInteger sommeDesParts = alloc.Values.Aggregate(BigInteger.Zero, (a, b) => a + b);
Console.WriteLine($"Efficacite : somme des parts = {sommeDesParts} == surplus grand coalition = {vSurplus} ? {sommeDesParts == vSurplus}");

Verifier(vSurplus == 23, "surplus optimal == 23 (= 4*9 - 13)");
Verifier(uMax == 6, "max(u) == 6 : translation valide a partir de U = 6");
Verifier(M - vMax == 9, "M - max(v) == 9 : borne superieure de l'intervalle de translation");
Verifier(coalitionsTestees == 254, "254 coalitions testees (2^8 - 2)");
Verifier(violations == 0, "aucune violation du coeur");
Verifier(sommeDesParts == 23 && sommeDesParts == vSurplus, "efficacite : somme des parts == surplus optimal == 23");

Surplus optimal : 23  (= n*M - 13 = 23)


Parts non negatives pour tout U dans [6, 9] ; on prend U = 6


Coalitions testees : 254


Violations du coeur par l'imputation KM : 0


Efficacite : somme des parts = 23 == surplus grand coalition = 23 ? True


Assertion OK : surplus optimal == 23 (= 4*9 - 13)


Assertion OK : max(u) == 6 : translation valide a partir de U = 6


Assertion OK : M - max(v) == 9 : borne superieure de l'intervalle de translation


Assertion OK : 254 coalitions testees (2^8 - 2)


Assertion OK : aucune violation du coeur


Assertion OK : efficacite : somme des parts == surplus optimal == 23


### Lecture du résultat

Surplus optimal **23 = 4·9 − 13** ; translation valide pour tout **U ∈ [6, 9]** (parts non
négatives) ; sur les **254 coalitions** ($2^{2n} - 2$), **0 violations** ; efficacité
exacte : la somme des parts (23) égale le surplus de la grande coalition. L'imputation dérivée
des potentiels KM est un point du cœur — le théorème de Shapley-Shubik, vérifié
exhaustivement sur cette instance.

### Ce que la vérification dit — et ne dit pas

L'imputation dérivée des potentiels $(u, v)$ de Kuhn-Munkres viole **aucune** des $2^{2n} - 2$
inégalités de coalition : elle est dans le cœur, et l'efficacité est réalisée. Ce que le test
numérique ne prouve pas, c'est la réciproque (tout le cœur est de cette forme) — c'est le
théorème de Shapley-Shubik, dont la formalisation Lean est livrée dans le lake compagnon
`assignment_lean/` (`Assignment/Optimality.lean`). Le lien LP ↔ cœur n'est pas une coïncidence :
c'est la **complémentarité primale-duale** qui, réinterprétée en langage de coalitions, dit
exactement qu'aucune coalition ne peut bloquer.

## 5. Gale-Shapley contre Kuhn-Munkres : deux notions d'optimum

Le dépôt contient déjà `game_theory_lean/StableMarriage/GaleShapley.lean` — le mariage stable
de Gale-Shapley. Les deux algorithmes répondent à des questions **différentes** sur le même type
d'objet (un matching) :

| | Gale-Shapley | Kuhn-Munkres |
|---|---|---|
| Données | préférences ordinales (rang) | coûts cardinaux |
| Objectif | **stabilité** (pas de paire bloquante) | **optimalité pondérée** (valeur min/max) |
| Garantie | stable, optimal pour le côté proposant | optimal en valeur, dualité certifiée |
| Complexité | $O(n^2)$ | $O(n^3)$ |

Sur notre matrice de coûts, Gale-Shapley ne s'applique pas tel quel (il veut des rangs) — mais
nous pouvons convertir les coûts en préférences et observer que le matching **stable** peut être
**plus cher** que l'optimum : stabilité et optimalité pondérée sont des propriétés indépendantes.

In [7]:
// Stabilite vs optimalite : Gale-Shapley sur preferences derivees des couts
// prefsAgents[i] = ordre de preference des taches (meilleur en premier).
// Retourne le matching (agent -> tache), optimal pour les AGENTS (cote proposant).
static int[] GaleShapley(int[][] prefsAgents, int[][] prefsTaches)
{
    int n = prefsAgents.Length;
    var libre = new Queue<int>();
    for (int i = 0; i < n; i++) libre.Enqueue(i);
    var propDe = new int[n];                          // prochain indice de proposition de chaque agent
    var tenue = new int[n]; Array.Fill(tenue, -1);    // tache j -> agent qui la tient actuellement
    var rangTache = new Dictionary<int, int>[n];
    for (int j = 0; j < n; j++)
    {
        rangTache[j] = new Dictionary<int, int>();
        for (int r = 0; r < n; r++) rangTache[j][prefsTaches[j][r]] = r;
    }
    while (libre.Count > 0)
    {
        int i = libre.Dequeue();
        int j = prefsAgents[i][propDe[i]];
        propDe[i]++;
        if (tenue[j] == -1) tenue[j] = i;
        else if (rangTache[j][i] < rangTache[j][tenue[j]])
        {
            int evince = tenue[j];
            tenue[j] = i;
            libre.Enqueue(evince);
        }
        else if (propDe[i] < n) libre.Enqueue(i);
    }
    var resultat = new int[n];
    for (int j = 0; j < n; j++) resultat[tenue[j]] = j;
    return resultat;
}

// Preferences = couts croissants (l'agent prefere la tache la MOINS chere pour lui ;
// tri stable : les egalites se cassent par l'indice, comme le tri du carnet Python).
static int[][] PreferencesLignes(BigInteger[,] M)
    => Enumerable.Range(0, M.GetLength(0))
        .Select(i => Enumerable.Range(0, M.GetLength(1)).OrderBy(j => M[i, j]).ToArray())
        .ToArray();
static int[][] PreferencesColonnes(BigInteger[,] M)
    => Enumerable.Range(0, M.GetLength(1))
        .Select(j => Enumerable.Range(0, M.GetLength(0)).OrderBy(i => M[i, j]).ToArray())
        .ToArray();

var prefsAgents = PreferencesLignes(C);
var prefsTaches = PreferencesColonnes(C);
int[] assignGs = GaleShapley(prefsAgents, prefsTaches);
BigInteger valGs = ValeurBrute(C, assignGs);
Console.WriteLine($"Gale-Shapley (stable)   : valeur = {valGs}, affectation = [{string.Join(", ", assignGs)}]");
Console.WriteLine($"Kuhn-Munkres (optimal)  : valeur = {valKm}, affectation = [{string.Join(", ", assignKm)}]");
Console.WriteLine($"Ecart sur l'exemple fil rouge : {valGs - valKm}");

Verifier(valGs == 13, "Gale-Shapley == 13 sur le fil rouge");
Verifier(valGs == valKm, "GS et KM coïncident sur le fil rouge (ecart 0)");

// Sur CETTE matrice, GS et KM coincident -- c'est possible, pas garanti. L'instance
// divergente du carnet Python (scan grille 7, essai 1) est rejouee telle quelle :
// CITE : matrice rapportee par la sortie commitee du carnet source Python.
var Mdiv = new BigInteger[4, 4]
{
    { 1, 7, 1, 4 },
    { 7, 2, 3, 2 },
    { 6, 2, 8, 4 },
    { 4, 4, 5, 4 },
};
var gsDiv = GaleShapley(PreferencesLignes(Mdiv), PreferencesColonnes(Mdiv));
var valGsDiv = ValeurBrute(Mdiv, gsDiv);
var (valKmDiv, _, _, _) = KuhnMunkres(Mdiv);
Console.WriteLine($"Instance divergente citee : GS stable = {valGsDiv}, KM optimal = {valKmDiv}");
Console.WriteLine($"La stabilite se paie : +{valGsDiv - valKmDiv} par rapport a l'optimum");

Verifier(valKmDiv == 9, "KM optimal == 9 sur l'instance citee");
Verifier(valGsDiv == 12, "GS stable == 12 sur l'instance citee");
Verifier(valGsDiv - valKmDiv == 3, "la stabilite se paie +3 sur l'instance citee");

// Le scan du jumeau C# (graine fixe) cherche d'autres divergences et rapporte honnetement.
var rng2 = new Random(7);
bool trouve = false;
for (int essai = 0; essai < 500 && !trouve; essai++)
{
    var Ms = new BigInteger[4, 4];
    for (int i = 0; i < 4; i++)
        for (int j = 0; j < 4; j++) Ms[i, j] = rng2.Next(0, 9);
    var gs = GaleShapley(PreferencesLignes(Ms), PreferencesColonnes(Ms));
    var vGs = ValeurBrute(Ms, gs);
    var vOpt = OptimumBrute(Ms);
    if (vGs > vOpt)
    {
        Console.WriteLine($"Le scan du jumeau C# trouve aussi (essai {essai}) : GS stable = {vGs}, KM optimal = {vOpt}");
        trouve = true;
    }
}
if (!trouve) Console.WriteLine("Le scan du jumeau C# n'a pas trouve de divergence en 500 essais -- rapporter honnetement");

Gale-Shapley (stable)   : valeur = 13, affectation = [1, 0, 2, 3]


Kuhn-Munkres (optimal)  : valeur = 13, affectation = [1, 0, 2, 3]


Ecart sur l'exemple fil rouge : 0


Assertion OK : Gale-Shapley == 13 sur le fil rouge


Assertion OK : GS et KM coïncident sur le fil rouge (ecart 0)


Instance divergente citee : GS stable = 12, KM optimal = 9


La stabilite se paie : +3 par rapport a l'optimum


Assertion OK : KM optimal == 9 sur l'instance citee


Assertion OK : GS stable == 12 sur l'instance citee


Assertion OK : la stabilite se paie +3 sur l'instance citee


Le scan du jumeau C# trouve aussi (essai 3) : GS stable = 11, KM optimal = 10


### Lecture du résultat

Sur le fil rouge, GS et KM **coïncident** (écart 0) — c'est possible, jamais garanti.
L'instance divergente rapportée par le carnet Python (scan graine 7, dès l'essai 1) est
rejouée telle quelle : une matrice 4×4 où le matching stable GS coûte **12** contre un optimum
KM de **9** — la stabilité se paie **+3**, soit 33 % au-dessus de l'optimum ; le scan seedé
propre au jumeau C# (graine 7, 500 essais) rapporte de son côté les divergences qu'il
rencontre. Stabilité ordinale et optimalité cardinale sont des propriétés **indépendantes** :
exiger l'une ne donne pas l'autre.

## 6. Exercices

Trois exercices pour s'approprier l'algorithme. Les stubs s'exécutent sans erreur (le notebook
reste exécutable de bout en bout) — à vous de les compléter.

### Exercice 1 — Détecter les affectations non optimales

Écrivez `EstOptimal(C, assignment)` qui vérifie (par comparaison à la force brute, possible
pour $n \le 6$) si une affectation donnée est optimale. Testez-la sur l'affectation identité
de la matrice $C$ du notebook — elle n'est probablement pas optimale.

In [8]:
// Exercice 1 : detecteur d'optimalite par force brute
// TODO etudiant
// Indice : reutiliser ValeurBrute et Permutations ; n = 4 ici, la methode tient pour n <= 6
// Etape 1 : calculer la valeur de l'affectation donnee
// Etape 2 : calculer l'optimum par force brute
// Etape 3 : comparer

static bool? EstOptimal(BigInteger[,] M, int[] assignment)
{
    // TODO etudiant
    return null;  // a completer : true ssi assignment atteint l'optimum
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — Le crédit partial du glouton

Le glouton ligne-par-ligne (chaque agent prend sa tâche la moins chère disponible) produit une
affectation **réalisable** mais sous-optimale sur certaines matrices. Implémentez le glouton,
trouvez (par recherche aléatoire seedée) une matrice 4×4 où il échoue, et mesurez l'écart à
l'optimum. Combien vaut le pire rapport valeur_glouton / valeur_optimale que vous observez ?

Exemple attendu : sur la matrice $\begin{pmatrix} 1 & 2 \\ 2 & 100 \end{pmatrix}$, le glouton
prend 1 puis 100, l'optimum est $2 + 2 = 4$.

In [9]:
// Exercice 2 : glouton ligne-par-ligne et son pire cas
// TODO etudiant
// Indice : pour chaque ligne dans l'ordre, prendre la colonne libre la moins chere
// Etape 1 : implementer Glouton(M) -> affectation realisable
// Etape 2 : balayer des matrices aleatoires seedees pour trouver glouton != optimum
// Etape 3 : mesurer le rapport valeur_glouton / valeur_optimale

static int[] Glouton(BigInteger[,] M)
{
    // TODO etudiant
    return null;  // a completer : une affectation realisable (une colonne libre par ligne)
}

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — Un deuxième point du cœur

Le cœur du jeu d'affectation est un **polytope**, pas un point : l'imputation de la section 4
dépend du choix de $U$ dans l'intervalle $[\max u,\ M - \max v]$. Choisissez un autre $U$,
vérifiez que la nouvelle imputation satisfait encore toutes les inégalités de coalition mais
donne des parts **différentes** à certains joueurs. Deux points du cœur = deux partages
équilibrés du même surplus optimal.

In [10]:
// Exercice 3 : un deuxieme point du coeur
// TODO etudiant
// Indice : reprendre la cellule de la section 4 avec un autre U, par exemple U2 = M - v.Max()
// Etape 1 : construire alloc2 par la translation alloc2_i = U2 - u_i, alloc2_{n+j} = (M - U2) - v_j
// Etape 2 : verifier part2(S) >= ValeurCoalition(S) pour toutes les coalitions
// Etape 3 : comparer alloc et alloc2 joueur par joueur (certaines parts changent, la somme non)

BigInteger U2 = uMax;  // TODO etudiant : choisir U2 dans [uMax, M - vMax], distinct de uMax
Console.WriteLine($"Exercice a completer : deuxieme point du coeur pour U2 dans [6, 9] (U2 = {U2} pour l'instant)");

Exercice a completer : deuxieme point du coeur pour U2 dans [6, 9] (U2 = 6 pour l'instant)


## 7. Conclusion

Ce notebook a suivi le fil d'un seul théorème — la dualité du problème d'affectation — à travers
trois lectures :

1. **Algorithmique** : Kuhn-Munkres from scratch, labels et arbres hongrois, confronté à la
force brute exhaustive (et, dans le jumeau Python, à la référence SOTA SciPy) — coïncidence
exacte sur l'exemple et 50 instances aléatoires.
2. **Optimisation linéaire** : les potentiels de l'algorithme sont une solution duale optimale,
le gap de dualité est nul, toutes les arêtes du matching sont des arêtes d'égalité —
c'est le certificat d'optimalité.
3. **Théorie des jeux** : le théorème de Shapley-Shubik — le cœur du jeu d'affectation est
exactement le polytope dual optimal. Vérifié numériquement : aucune coalition ne bloque
l'imputation, la valeur est épuisée. La formalisation Lean vit dans `assignment_lean/`.

Et la comparaison Gale-Shapley / Kuhn-Munkres rappelle que « bon matching » n'a pas un sens
unique : stabilité et optimalité pondérée sont des propriétés logiquement indépendantes.

## Références

- Kuhn, H. W. (1955). « The Hungarian Method for the Assignment Problem ». *Naval Research
  Logistics Quarterly* 2:83–97.
- Munkres, J. R. (1957). « Algorithms for the Assignment and Transportation Problems ».
  *J. SIAM* 5(1):32–38 — l'analyse de complexité fortement polynomiale.
- Shapley, L. S. & Shubik, M. (1971). « The Assignment Game I: The Core ». *International
  Journal of Game Theory* 1:111–130.
- `scipy.optimize.linear_sum_assignment` — la référence SOTA du jumeau Python
  (Jonker-Volgenant) ; le jumeau C# confronte la force brute interne, exacte par construction.
- Hommage : James R. Munkres (1932–2026) — voir l'en-tête du carnet.